# M3: WangchanBERTa fine-tune (รันบน Google Colab)

ก่อนเริ่ม: **Runtime → Change runtime type → T4 GPU**

โน้ตบุ๊กนี้แค่เรียก `train_m3.py` — โค้ดจริงอยู่ใน repo (`src/transformer.py`, `train_m3.py`) ให้ผลที่ได้อยู่ในรูปแบบเดียวกับ B0/B1

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1) เอาโค้ดเข้ามา
เลือกวิธีใดวิธีหนึ่ง
- **ก. clone จาก GitHub** (ต้อง push `train_m3.py`, `src/transformer.py`, `src/config.py` ขึ้นไปก่อน; ถ้า repo เป็น private ให้ใช้ `https://<token>@github.com/...`)
- **ข. อัปโหลด zip** ของโฟลเดอร์ `thai-toxic-detection` (ไม่ต้องใส่ `venv_deepcut/`) ผ่านแถบ Files ด้านซ้าย แล้วรันเซลล์ ข.

In [ ]:
# ก. clone
!git clone https://github.com/KhunPutcharapon/thai-toxic-detection.git
%cd /content/thai-toxic-detection

In [ ]:
# ข. unzip (ถ้าใช้วิธีอัปโหลด)
# !unzip -q /content/thai-toxic-detection.zip -d /content/
# %cd /content/thai-toxic-detection

## 2) ติดตั้งไลบรารี
Colab มี torch, scikit-learn, pandas, scipy อยู่แล้ว เพิ่มแค่ transformers + sentencepiece (tokenizer ของ WangchanBERTa)

In [ ]:
!pip -q install transformers sentencepiece
import torch, transformers, sklearn
print('torch', torch.__version__, '| transformers', transformers.__version__, '| sklearn', sklearn.__version__)

## 3) Smoke test (~1–2 นาที)
เช็กว่าโหลดโมเดล/ข้อมูล/บันทึกผลได้ ตัวเลขที่ได้ไม่มีความหมาย

In [ ]:
!python train_m3.py --smoke

## 4) รันจริง (~15–25 นาที บน T4)
เลือก lr/epoch จาก dev 10% → ฝึกใหม่บนชุดฝึกทั้งหมด 5 seeds → ทายชุดทดสอบ 176 ทวีต

In [ ]:
!python train_m3.py 2>&1 | tee results_m3_log.txt

## 5) เทียบกับ B1 และ B0 (McNemar + bootstrap)

In [ ]:
!python compare.py --a results/b1 --b results/m3
!python compare.py --a results/b0 --b results/m3

## 6) ดาวน์โหลดผล
แตก zip แล้ววางโฟลเดอร์ `results/m3/` และไฟล์ `results/compare_*_vs_m3.json` ในโปรเจกต์บนเครื่อง แล้ว commit

In [ ]:
!cp results_m3_log.txt results/m3/train_log.txt
!zip -qr m3_results.zip results/m3 results/compare_b1_vs_m3.json results/compare_b0_vs_m3.json
from google.colab import files
files.download('m3_results.zip')